# SMS spam classifier — training and backend artifact

This notebook follows the attached notebook's flow (load data → inspect → split → vectorize → train → evaluate), while making the deployment boundary explicit. The exported `Pipeline` owns the text representation and classifier together, so the backend does not need to duplicate preprocessing.

The default model is word + character TF-IDF with logistic regression: a compact, low-latency baseline for short SMS messages.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, roc_auc_score
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'notebooks' / 'SMS' / 'train.py').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
SMS_DIR = PROJECT_ROOT / 'notebooks' / 'SMS'
sys.path.insert(0, str(SMS_DIR))
from train import load_sms_csv, make_pipeline

DATA_PATH = PROJECT_ROOT / 'data' / 'processed' / 'messages' / 'uci_messages_cleaned.csv'
ARTIFACT_DIR = PROJECT_ROOT / 'ml' / 'artifacts' / 'sms_spam'

## 1. Load and inspect the labeled messages

Accepted columns are `v1/v2` from the SMS Spam Collection or `target/message`. Keep the original message intact; the vectorizers handle casing and character patterns.

In [ ]:
df = load_sms_csv(DATA_PATH)
display(df.head())
display(df['target'].value_counts().rename_axis('label').to_frame('count'))
df['message_length'] = df['message'].str.split().str.len()
display(df.groupby('target')['message_length'].describe().round(1))

In [ ]:
df.groupby('target')['message_length'].plot(kind='hist', alpha=.55, bins=30, legend=True)
plt.xlabel('words per message'); plt.show()

## 2. Split before fitting any representation

The split is stratified and fixed for repeatability. Vectorizer vocabularies are learned only from the training partition to avoid leakage.

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(
    df['message'], df['target'], test_size=.20, random_state=42, stratify=df['target']
)
pipeline = make_pipeline()
pipeline.fit(x_train, y_train)

## 3. Evaluate the decision the product actually needs

Accuracy alone is not enough. Inspect spam precision/recall, ROC-AUC, and the confusion matrix. Tune the threshold against the cost of letting spam through versus incorrectly hiding a legitimate SMS.

In [ ]:
pred = pipeline.predict(x_test)
spam_index = list(pipeline.classes_).index('spam')
spam_probability = pipeline.predict_proba(x_test)[:, spam_index]
print(classification_report(y_test, pred))
print('ROC-AUC:', round(roc_auc_score((y_test == 'spam').astype(int), spam_probability), 4))
ConfusionMatrixDisplay.from_predictions(y_test, pred, labels=['ham', 'spam'])
plt.show()

In [ ]:
# Example threshold sweep; choose using validation/product costs, not test-set optimization.
from sklearn.metrics import precision_score, recall_score
for threshold in (.30, .40, .50, .60, .70):
    threshold_pred = ['spam' if p >= threshold else 'ham' for p in spam_probability]
    print(threshold, 'precision=', round(precision_score(y_test, threshold_pred, pos_label='spam'), 3),
          'recall=', round(recall_score(y_test, threshold_pred, pos_label='spam'), 3))

## 4. Export the backend artifacts

The pipeline is the deployable model. Metadata records the version, threshold, labels, and evaluation so the backend and monitoring system can identify what produced a prediction.

In [ ]:
# This calls the same export function used by the command-line training job.
from train import train
metadata = train(str(DATA_PATH), str(ARTIFACT_DIR))
print('Saved:', ARTIFACT_DIR / 'sms_spam_pipeline.joblib')
print('Saved:', ARTIFACT_DIR / 'sms_spam_metadata.json')

## 5. Smoke-test the exact backend interface

Load the artifact in a fresh object and test representative ham, spam, short, long, URL, emoji, and multilingual messages before deploying.

In [ ]:
from predict import SmsClassifier
classifier = SmsClassifier(ARTIFACT_DIR)
for message in [
    'Can you pick up milk on your way home?',
    'Congratulations! You won a free prize. Call now!',
]:
    print(classifier.predict(message))